In [1]:
%pip install qi

Note: you may need to restart the kernel to use updated packages.


In [3]:
import qi
ip = "169.254.199.30"
port = 9559
session = qi.Session()
session.connect("tcp://" + ip + ":" + str(port))

In [ ]:

#ex 1 presente en public 
tts = session.service("ALTextToSpeech")
tts.say("Bonjour , je fonctionne avec Python 3 et qi.")


In [ ]:
#ex2
tts = session.service("ALTextToSpeech")
motion = session.service("ALMotion")
posture = session.service("ALRobotPosture")

# 1. Réveiller le robot et le mettre debout
motion.wakeUp()
posture.goToPosture("StandInit", 0.5)

# 2. Lever le bras droit
joints = ["RShoulderPitch", "RShoulderRoll", "RElbowYaw", "RElbowRoll"]
angles = [-1.0, -0.3, 0.5, 1.0]
motion.angleInterpolation(joints, angles, 1.5, True)

# 3. Parler pendant que le bras salue (_async=True : le robot parle sans bloquer)
tts.say("Bonjour . Je suis un robot NAO.", _async=True)

# 4. Agiter l'avant-bras : le coude va et vient
for i in range(3):
    motion.angleInterpolation("RElbowRoll", 1.5, 0.4, True)
    motion.angleInterpolation("RElbowRoll", 0.5, 0.4, True)

# 5. Revenir en position debout normale
posture.goToPosture("StandInit", 0.5)


False

In [16]:
#ex3
import time


memory = session.service("ALMemory")
tts = session.service("ALTextToSpeech")

capteurs = {
    # Tête
    "Device/SubDeviceList/Head/Touch/Front/Sensor/Value": "avant de la tête",
    "Device/SubDeviceList/Head/Touch/Middle/Sensor/Value": "milieu de la tête",
    "Device/SubDeviceList/Head/Touch/Rear/Sensor/Value": "arrière de la tête",

    # Main gauche
    "Device/SubDeviceList/LHand/Touch/Left/Sensor/Value": "côté gauche de la main gauche",
    "Device/SubDeviceList/LHand/Touch/Back/Sensor/Value": "dos de la main gauche",
    "Device/SubDeviceList/LHand/Touch/Right/Sensor/Value": "côté droit de la main gauche",

    # Main droite
    "Device/SubDeviceList/RHand/Touch/Left/Sensor/Value": "côté gauche de la main droite",
    "Device/SubDeviceList/RHand/Touch/Back/Sensor/Value": "dos de la main droite",
    "Device/SubDeviceList/RHand/Touch/Right/Sensor/Value": "côté droit de la main droite",
}

etat_precedent = {nom: 0 for nom in capteurs}

print("Touchez un capteur...")

while True:
    for nom_capteur, phrase in capteurs.items():
        try:
            valeur = memory.getData(nom_capteur)

            if valeur > 0.5 and etat_precedent[nom_capteur] <= 0.5:
                print("Touché :", phrase)
                tts.say(phrase)

            etat_precedent[nom_capteur] = valeur

        except Exception as e:
            print("Indisponible :", nom_capteur)

    time.sleep(0.1)

Touchez un capteur...
Touché : avant de la tête
Touché : dos de la main droite


KeyboardInterrupt: 

In [ ]:
#ex4
memory = session.service("ALMemory")
tts = session.service("ALTextToSpeech")

capteurs = {
    # Tête
    "Device/SubDeviceList/Head/Touch/Front/Sensor/Value": "avant de la tête",
    "Device/SubDeviceList/Head/Touch/Middle/Sensor/Value": "milieu de la tête",
    "Device/SubDeviceList/Head/Touch/Rear/Sensor/Value": "arrière de la tête",

    # Main gauche
    "Device/SubDeviceList/LHand/Touch/Left/Sensor/Value": "côté gauche de la main gauche",
    "Device/SubDeviceList/LHand/Touch/Back/Sensor/Value": "dos de la main gauche",
    "Device/SubDeviceList/LHand/Touch/Right/Sensor/Value": "côté droit de la main gauche",

    # Main droite
    "Device/SubDeviceList/RHand/Touch/Left/Sensor/Value": "côté gauche de la main droite",
    "Device/SubDeviceList/RHand/Touch/Back/Sensor/Value": "dos de la main droite",
    "Device/SubDeviceList/RHand/Touch/Right/Sensor/Value": "côté droit de la main droite",
}

etat_precedent = {nom: 0 for nom in capteurs}

print("Touchez un capteur...")

while True:
    for nom_capteur, phrase in capteurs.items():
        try:
            valeur = memory.getData(nom_capteur)

            if valeur > 0.5 and etat_precedent[nom_capteur] <= 0.5:
                print("Touché :", phrase)
                tts.say(phrase)

            etat_precedent[nom_capteur] = valeur

        except Exception as e:
            print("Indisponible :", nom_capteur)

    time.sleep(0.1)

In [ ]:
#ex5
memory = session.service("ALMemory")
tts = session.service("ALTextToSpeech")

head_sensor = "FrontTactilTouched"

compteur = 0
etat_precedent = 0

print("Touchez la tête 3 fois...")

while True:
    valeur = memory.getData(head_sensor)

    # Détecter seulement le début d'un nouveau toucher
    if valeur > 0 and etat_precedent == 0:
        compteur += 1
        print("Nombre de touches :", compteur)

        if compteur == 3:
            tts.say("Bonjour")
            print("Pepper dit : Bonjour")

            # Optionnel : remettre le compteur à zéro
            compteur = 0

    etat_precedent = valeur

    time.sleep(0.1)

In [ ]:
#ex 6
motion = session.service("ALMotion")
posture = session.service("ALRobotPosture")

# Mettre Pepper debout
posture.goToPosture("StandInit", 0.5)

# Avancer de 1 mètre
motion.moveTo(1.0, 0.0, 0.0)

print("Pepper a avancé de 1 mètre")

In [ ]:
#ex 7 
motion = session.service("ALMotion")
memory = session.service("ALMemory")

# Capteur avant de la tête
head_sensor = "FrontTactilTouched"

etat_precedent = 0
en_mouvement = True

# Réveiller Pepper
motion.wakeUp()

print("Pepper avance")
print("Touchez la tête pour arrêter / reprendre")

# Commence à avancer
motion.moveToward(0.2, 0.0, 0.0)

while True:
    valeur = memory.getData(head_sensor)

    # Nouveau toucher détecté
    if valeur > 0 and etat_precedent == 0:

        if en_mouvement:
            # Arrêt
            motion.stopMove()
            en_mouvement = False
            print("Pepper arrêté")

        else:
            # Reprendre
            motion.moveToward(0.2, 0.0, 0.0)
            en_mouvement = True
            print("Pepper avance")

    etat_precedent = valeur
    time.sleep(0.1)

In [ ]:
#ex8
motion = session.service("ALMotion")
memory = session.service("ALMemory")

# Capteur avant de la tête
head_sensor = "FrontTactilTouched"

etat_precedent = 0
en_mouvement = True

# Réveiller Pepper
motion.wakeUp()

print("Pepper avance")
print("Touchez la tête pour arrêter / reprendre")

# Commence à avancer
motion.moveToward(0.2, 0.0, 0.0)

while True:
    valeur = memory.getData(head_sensor)

    # Nouveau toucher détecté
    if valeur > 0 and etat_precedent == 0:

        if en_mouvement:
            # Arrêt
            motion.stopMove()
            en_mouvement = False
            print("Pepper arrêté")

        else:
            # Reprendre
            motion.moveToward(0.2, 0.0, 0.0)
            en_mouvement = True
            print("Pepper avance")

    etat_precedent = valeur
    time.sleep(0.1)

In [ ]:
#ex9
import math


motion = session.service("ALMotion")

motion.wakeUp()

angle = 2 * math.pi / 3

# Triangle
for i in range(3):
    print("Côté", i + 1)

    motion.moveToward(0.15, 0.0, 0.0)
    time.sleep(2)
    motion.stopMove()

    motion.moveTo(0.0, 0.0, angle)

print("Triangle terminé")

# Attendre 3 secondes
time.sleep(3)

# Simuler une posture de fin adaptée à Pepper
motion.setAngles(
    ["HeadPitch", "LShoulderPitch", "RShoulderPitch"],
    [0.4, 1.3, 1.3],
    0.2
)

print("Posture finale")

In [ ]:
#ex10
motion = session.service("ALMotion")
memory = session.service("ALMemory")

motion.wakeUp()

# Exemple de clé capteur de distance frontal
distance_sensor = "Device/SubDeviceList/Platform/Front/Sonar/Sensor/Value"

print("Déplacement avec évitement d'obstacle")

while True:
    try:
        distance = memory.getData(distance_sensor)

        print("Distance :", distance)

        # Si obstacle proche
        if distance < 0.5:
            print("Obstacle détecté")

            # Stop
            motion.stopMove()

            # Tourner
            motion.moveTo(0.0, 0.0, 1.0)

            # S'éloigner un peu
            motion.moveToward(0.15, 0.0, 0.0)
            time.sleep(1.5)
            motion.stopMove()

        else:
            # Avancer doucement
            motion.moveToward(0.1, 0.0, 0.0)

    except Exception as e:
        print("Capteur indisponible :", e)
        motion.stopMove()
        break

    time.sleep(0.1)

In [22]:
#ex11

import random


tts = session.service("ALTextToSpeech")

mots = ["Bonjour", "Salut", "Bienvenue", "Robotique"]

mot_choisi = random.choice(mots)

print("Mot choisi :", mot_choisi)

tts.say(mot_choisi)

Mot choisi : Bienvenue


In [24]:
#ex12

memory = session.service("ALMemory")
tts = session.service("ALTextToSpeech")
motion = session.service("ALMotion")

# Capteurs
left_hand = "Device/SubDeviceList/LHand/Touch/Back/Sensor/Value"
head = "Device/SubDeviceList/Head/Touch/Front/Sensor/Value"
right_hand = "Device/SubDeviceList/RHand/Touch/Back/Sensor/Value"

# Mot de passe attendu
password = [
    "main_gauche",
    "main_gauche",
    "tete",
    "main_droite"
]

sequence = []

previous_left = 0
previous_head = 0
previous_right = 0

print("Entrez le mot de passe physique...")

while True:

    left = memory.getData(left_hand)
    head_value = memory.getData(head)
    right = memory.getData(right_hand)

    # Main gauche
    if left > 0.5 and previous_left <= 0.5:
        sequence.append("main_gauche")
        print("Main gauche")

    # Tête
    if head_value > 0.5 and previous_head <= 0.5:
        sequence.append("tete")
        print("Tête")

    # Main droite
    if right > 0.5 and previous_right <= 0.5:
        sequence.append("main_droite")
        print("Main droite")

    previous_left = left
    previous_head = head_value
    previous_right = right

    # Quand 4 touches ont été effectuées
    if len(sequence) == 4:

        print("Séquence :", sequence)

        if sequence == password:

            print("Mot de passe correct")
            tts.say("Mot de passe correct. Je suis déverrouillé.")

            # Petit geste pour montrer le déverrouillage
            motion.setAngles(
                ["RShoulderPitch", "RElbowRoll"],
                [-0.5, 1.0],
                0.2
            )

            time.sleep(2)

        else:

            print("Mot de passe incorrect")
            tts.say("Mot de passe incorrect")

        # Recommencer une nouvelle tentative
        sequence = []

    time.sleep(0.1)

Entrez le mot de passe physique...
Tête
Main droite
Main gauche
Tête
Séquence : ['tete', 'main_droite', 'main_gauche', 'tete']
Mot de passe incorrect
Tête
Tête


KeyboardInterrupt: 